In [19]:
%load_ext autoreload
%autoreload 2

In [20]:
import contextlib
import copy
import io
import os
import queue
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from sklearn.metrics import root_mean_squared_error, r2_score

from aquacrop import AquaCrop, Crop, Soil, Weather
from aquacrop_slovenia import config
from aquacrop_slovenia.diagnostics import nse, kge, mkge, kge_r, kge_beta, kge_alpha, mkge_alpha
from aquacrop_slovenia.plots import plot_yield_timeseries_comparison, plot_yield_scatter, plot_yield_timeseries_residuals
from aquacrop_slovenia.reading_data import get_yield, get_yield_for_comparison, get_co2_for_aquacrop, get_station_weather
from aquacrop_slovenia.parameter_defaults_rakican import (
    rakican_soil_layers,
    rakican_curve_number,
    rakican_readily_evaporable_water,
    rakican_maize_params,
    rakican_optimal_management,
    rakican_initial_cond,
    rakican_groundwater,
)

In [21]:
LOCATION = "rakican"
MAX_WORKERS = os.cpu_count()

simulation_periods = [
    {
        "start_date": date(year, 1, 1),
        "end_date": date(year, 12, 31),
        "planting_date": date(year, 4, 20),
        "is_seeding_year": True,
    }
    for year in range(1993, 2024)
    if year not in [1998, 2023, 2017]
]

temperatures, eto, rain = get_station_weather(355)
historical_co2 = get_co2_for_aquacrop("historical")

weather = Weather(
    location=LOCATION,
    temperatures=temperatures,
    eto_values=eto,
    rainfall_values=rain,
    record_type=1,
    first_day=1,
    first_month=1,
    first_year=1993,
    co2_records=historical_co2,
)

soil = Soil(
    name=f"{LOCATION} soil",
    description=f"{LOCATION} loamy sand soil",
    soil_layers=rakican_soil_layers,
    curve_number=rakican_curve_number,
    readily_evaporable_water=rakican_readily_evaporable_water,
)

observed_biomass = get_yield_for_comparison(LOCATION, "biomass", "A", "N3")
observed_grain   = get_yield_for_comparison(LOCATION, "grain",   "A", "N3")

INTEGER_PARAMS = {name for name, val in rakican_maize_params.items() if isinstance(val, int)}

In [22]:
worker_dirs = []
for i in range(MAX_WORKERS):
    d = config.RAMDISK_DIR / f"de_rakican_{i}"
    d.mkdir(parents=True, exist_ok=True)
    worker_dirs.append(d)

dir_queue = queue.Queue()
for d in worker_dirs:
    dir_queue.put(d)

print(f"Worker directories ready: {MAX_WORKERS}")

In [23]:
# Parameter names and search bounds
# Defaults: harvest_index=0.51, gdd_senescence=1193, gdd_maturity=1466,
#           gdd_flowering=660, gdd_flowering_length=164, max_canopy_cover=0.90
param_names = [
    "harvest_index",
    "gdd_senescence",
    "gdd_maturity",
    "gdd_flowering",
    "gdd_flowering_length",
    "max_canopy_cover",
]

bounds = [
    (0.44, 0.58),   # harvest_index
    (1050, 1350),   # gdd_senescence
    (1300, 1600),   # gdd_maturity
    (580,  740),    # gdd_flowering
    (120,  200),    # gdd_flowering_length
    (0.82, 0.95),   # max_canopy_cover
]

print(f"Parameters: {param_names}")
print(f"Bounds:     {bounds}")

In [24]:
def evaluate_params(param_values):
    """Run one candidate and return all metrics for both grain and biomass."""
    working_dir = dir_queue.get()
    try:
        params = copy.deepcopy(rakican_maize_params)
        for name, val in zip(param_names, param_values):
            params[name] = int(round(val)) if name in INTEGER_PARAMS else float(val)

        crop = Crop(
            name=f"{LOCATION} maize",
            description="differential evolution",
            params=params,
        )

        simulation = AquaCrop(
            simulation_periods=simulation_periods,
            crop=crop,
            soil=soil,
            management=rakican_optimal_management,
            initial_conditions=rakican_initial_cond,
            climate=weather,
            #ground_water=rakican_groundwater,
            working_dir=working_dir,
            need_daily_output=False,
            need_seasonal_output=True,
            need_harvest_output=False,
            need_evaluation_output=False,
        )

        nan_metrics = {
            f"{prefix}_{s}": np.nan
            for prefix in ("biomass", "grain")
            for s in ["rmse", "r2", "nse", "kge", "mkge", "kge_r", "kge_beta", "kge_alpha", "mkge_alpha"]
        }

        try:
            with contextlib.redirect_stdout(io.StringIO()):
                results = simulation.run()
        except Exception:
            return nan_metrics

        seasonal = results["season"][["Year1", "BioMass", "Y(dry)"]].rename(
            columns={"Year1": "year", "BioMass": "biomass_modeled", "Y(dry)": "grain_modeled"}
        )

        def compute_metrics(observed_df, modeled_col, prefix):
            merged = observed_df.merge(seasonal[["year", modeled_col]], on="year")
            y_obs = merged["yield"].values
            y_mod = merged[modeled_col].values
            return {
                f"{prefix}_rmse":       root_mean_squared_error(y_obs, y_mod),
                f"{prefix}_r2":         r2_score(y_obs, y_mod),
                f"{prefix}_nse":        nse(y_mod, y_obs),
                f"{prefix}_kge":        kge(y_mod, y_obs),
                f"{prefix}_mkge":       mkge(y_mod, y_obs),
                f"{prefix}_kge_r":      kge_r(y_mod, y_obs),
                f"{prefix}_kge_beta":   kge_beta(y_mod, y_obs),
                f"{prefix}_kge_alpha":  kge_alpha(y_mod, y_obs),
                f"{prefix}_mkge_alpha": mkge_alpha(y_mod, y_obs),
            }

        return {
            **compute_metrics(observed_biomass, "biomass_modeled", "biomass"),
            **compute_metrics(observed_grain,   "grain_modeled",   "grain"),
        }
    finally:
        dir_queue.put(working_dir)

In [25]:
eval_log = []          # collects every evaluated candidate for post-analysis
eval_log_lock = threading.Lock()

def make_scalar_objective(objective_fn):
    """Scalar objective for one candidate (used by thread_map workers)."""
    def scalar(param_values):
        r = evaluate_params(param_values)
        with eval_log_lock:
            eval_log.append({**dict(zip(param_names, param_values)), **r})
        return objective_fn(r)
    return scalar

def thread_map(func, iterable):
    """Drop-in map replacement for scipy workers= that uses ThreadPoolExecutor."""
    items = list(iterable)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(func, items))


mean_biomass = observed_biomass["yield"].mean()
mean_grain   = observed_grain["yield"].mean()
print(f"Mean observed biomass: {mean_biomass:.1f}  |  mean observed grain: {mean_grain:.1f}")

objectives = {
    "biomass_rmse":   make_scalar_objective(lambda r: r["biomass_rmse"]),
    "grain_rmse":     make_scalar_objective(lambda r: r["grain_rmse"]),
    "combined_nrmse": make_scalar_objective(
        lambda r: r["biomass_rmse"] / mean_biomass + r["grain_rmse"] / mean_grain
    ),
}

In [26]:
DE_SETTINGS = dict(
    bounds=bounds,
    strategy="best1bin",
    maxiter=10,
    popsize=5,          # 15 * n_params = 90 individuals per generation
    tol=1e-4,
    mutation=(0.5, 1.0),
    recombination=0.7,
    seed=42,
    workers=thread_map,  # parallel via ThreadPoolExecutor; evaluates full generation at once
    updating="deferred", # required when workers != 1
    disp=True,
)

n_pop = DE_SETTINGS["popsize"] * len(bounds)
print(f"Population size: {n_pop}  |  max evaluations: {n_pop * DE_SETTINGS['maxiter']:,}")

In [28]:
eval_log.clear()
print("=== Optimising for BIOMASS RMSE ===")
result_biomass = differential_evolution(objectives["biomass_rmse"], **DE_SETTINGS)
print(f"\nSuccess: {result_biomass.success}  |  message: {result_biomass.message}")
print(f"Function evaluations: {result_biomass.nfev}")
print(f"Best biomass RMSE: {result_biomass.fun:.4f}")
print("Best parameters:")
for name, val in zip(param_names, result_biomass.x):
    print(f"  {name}: {val:.4f}")

RuntimeError: The map-like callable must be of the form f(func, iterable), returning a sequence of numbers the same length as 'iterable'

In [ ]:
eval_log.clear()
print("=== Optimising for GRAIN RMSE ===")
result_grain = differential_evolution(objectives["grain_rmse"], **DE_SETTINGS)
print(f"\nSuccess: {result_grain.success}  |  message: {result_grain.message}")
print(f"Function evaluations: {result_grain.nfev}")
print(f"Best grain RMSE: {result_grain.fun:.4f}")
print("Best parameters:")
for name, val in zip(param_names, result_grain.x):
    print(f"  {name}: {val:.4f}")

In [ ]:
eval_log.clear()
print("=== Optimising for COMBINED normalised RMSE (biomass + grain) ===")
result_combined = differential_evolution(objectives["combined_nrmse"], **DE_SETTINGS)
print(f"\nSuccess: {result_combined.success}  |  message: {result_combined.message}")
print(f"Function evaluations: {result_combined.nfev}")
print(f"Best combined nRMSE: {result_combined.fun:.6f}")
print("Best parameters:")
for name, val in zip(param_names, result_combined.x):
    print(f"  {name}: {val:.4f}")

In [ ]:
# Re-evaluate each best solution to get the full metric set
de_runs = {
    "biomass_rmse":   result_biomass.x,
    "grain_rmse":     result_grain.x,
    "combined_nrmse": result_combined.x,
}

summary_rows = []
for run_name, x in de_runs.items():
    metrics = evaluate_params(x)
    row = {"optimised_for": run_name, **dict(zip(param_names, x)), **metrics}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("optimised_for")
summary_df

In [ ]:
display_cols = [
    "biomass_rmse", "biomass_r2", "biomass_nse", "biomass_kge", "biomass_mkge",
    "grain_rmse",   "grain_r2",   "grain_nse",   "grain_kge",   "grain_mkge",
]
print("Metric comparison across optimisation targets:")
summary_df[display_cols].round(4)

In [ ]:
# Collect all evaluated candidates from the last DE run (combined) into a DataFrame
all_evals_df = pd.DataFrame(eval_log)
print(f"Total evaluations stored from last run: {len(all_evals_df)}")
all_evals_df.sort_values("biomass_rmse").head(10)

In [ ]:
output_path = config.RESULTS_DIR / "de_rakican_summary.pkl"
summary_df.reset_index().to_pickle(output_path)
print(f"Saved summary ({len(summary_df)} rows) to {output_path}")

In [ ]:
def rerun_best(param_values, label):
    """Re-run a full simulation with given params and return seasonal results."""
    params = copy.deepcopy(rakican_maize_params)
    for name, val in zip(param_names, param_values):
        params[name] = int(round(val)) if name in INTEGER_PARAMS else float(val)

    best_dir = config.RAMDISK_DIR / f"de_rakican_best_{label}"
    best_dir.mkdir(parents=True, exist_ok=True)

    sim = AquaCrop(
        simulation_periods=simulation_periods,
        crop=Crop(name=f"{LOCATION} maize ({label})", description=label, params=params),
        soil=soil,
        management=rakican_optimal_management,
        initial_conditions=rakican_initial_cond,
        climate=weather,
        #ground_water=rakican_groundwater,
        working_dir=best_dir,
        need_daily_output=False,
        need_seasonal_output=True,
        need_harvest_output=False,
        need_evaluation_output=False,
    )
    return sim.run()["season"]

In [ ]:
seasonal_biomass_opt = rerun_best(result_biomass.x, "biomass_opt")
print("=== Best solution optimised for biomass RMSE ===")
plot_yield_timeseries_comparison(seasonal_biomass_opt, get_yield(LOCATION, "biomass", "A", "N3"), "BioMass", LOCATION)

In [ ]:
seasonal_grain_opt = rerun_best(result_grain.x, "grain_opt")
print("=== Best solution optimised for grain RMSE ===")
plot_yield_timeseries_comparison(seasonal_grain_opt, get_yield(LOCATION, "grain", "A", "N3"), "Y(dry)", LOCATION)

In [ ]:
seasonal_combined_opt = rerun_best(result_combined.x, "combined_opt")
print("=== Best solution optimised for combined nRMSE ===")
plot_yield_timeseries_comparison(seasonal_combined_opt, get_yield(LOCATION, "biomass", "A", "N3"), "BioMass", LOCATION)

In [ ]:
plot_yield_timeseries_comparison(seasonal_combined_opt, get_yield(LOCATION, "grain", "A", "N3"), "Y(dry)", LOCATION)